In [ ]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS
# ==============================================================================

# Ferramentas de Nuvem (Google Cloud Platform - GCP)
from google.cloud import bigquery  # Conector oficial do banco de dados analítico (BigQuery) da Google.
from google.oauth2 import service_account  # Mecanismo de autenticação por chave/arquivo JSON de conta de serviço.

# Manipulação e Análise de Dados
import pandas as pd  # A principal biblioteca de Data Science para tabelas (DataFrames) e manipulação matricial.

# ==============================================================================
# 2. CONFIGURAÇÕES DE VISUALIZAÇÃO (PANDAS DISPLAY CONFIG)
# ==============================================================================

# Por padrão, o Pandas esconde colunas/linhas se a tabela for muito grande.
# Estas configurações garantem que possamos inspecionar os dados no terminal/notebook.
pd.set_option("display.max_columns", None)  # Não limita o número máximo de colunas exibidas na tela.
pd.set_option("display.max_rows", 10)  # Limita a exibição visual a no máximo 10 linhas para não travar a IDE.

In [ ]:
# ==============================================================================
# 3. CONFIGURAÇÃO DE FONTES DE DADOS E INFRAESTRUTURA DE NUVEM
# ==============================================================================

# --- FONTE 1: REPOSITÓRIO REMOTO (GITHUB) ---
# URL base onde estão armazenados arquivos CSV públicos contendo dados históricos de filmes.
url_github_movies_dataset = "https://media.githubusercontent.com/media/lucas-aulas/dataset-movies/refs/heads/main"

# --- FONTE 2: AMBIENTE DE LEITURA (PROJETO COMPARTILHADO DO PROFESSOR) ---
# ID do projeto na Google Cloud de onde leremos uma tabela institucional de bilheterias.
fonte_gcp_project_id = "meu-projeto-eng-dados"

# --- DESTINO: AMBIENTE DE ESCRITA (PROJETO PRIVADO DO ALUNO) ---
# Caminho local do arquivo JSON que contém as credenciais de segurança (chave privada).
# ATENÇÃO: Nunca envie esta chave para o GitHub público! Ela equivale à senha do seu banco de dados.
meu_service_account_file = "/home/user/Desktop/GitHub/axios/include/assets/gcp-sa-eanalytics-01.json"
meu_gcp_project_id = "meu-projeto-eng-dados"  # ID do seu projeto pessoal criado no GCP.

# Inicialização da autenticação e do cliente BigQuery
# Transforma o arquivo JSON em um objeto de credenciais aceito pelo Google.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# O objeto 'client' funciona como a nossa ponte ativa de conexão/comunicação com a nuvem do Google.
bigquery_client = bigquery.Client(credentials=gcp_credentials)

In [ ]:
df_imdb_filmes = pd.read_csv(f"{url_github_movies_dataset}/imdb_filmes.csv")

In [ ]:
df_box_office_dc_marvel = bigquery_client.query(
    f"SELECT * FROM {fonte_gcp_project_id}.raw.boxoffice_dc_marvel"
).to_dataframe(create_bqstorage_client=False)

In [ ]:
# ==============================================================================
# 7. CARGA INICIAL - CAMADA BRONZE (LOAD)
# ==============================================================================

# Enviando os dados brutos locais para tabelas físicas dentro do BigQuery.
# 'destination_table="bronze.nome"' cria ou insere os dados no Dataset chamado 'bronze'.
# 'if_exists="replace"' indica que se a tabela já existir lá em cima, ela será apagada e recriada do zero.
df_imdb_filmes.to_gbq(
    project_id=meu_gcp_project_id,
    destination_table="bronze.imdb_filmes",
    if_exists="replace",
    credentials=gcp_credentials,
)

df_boxoffice_dc_marvel.to_gbq(
    project_id=meu_gcp_project_id,
    destination_table="bronze.boxoffice_dc_marvel",
    if_exists="replace",
    credentials=gcp_credentials,
)